In [114]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [116]:
#
# load final X and y arrays from 'final_X_and_y.npz'
#

from pathlib import Path
import pickle

directory_path_input = Path("D:/ECG/input_for_model") # chose the right path beforhand
loaded_arrays = np.load(directory_path_input/'X_y_input.npz', allow_pickle=True)

# chose the file to restore:
X_train = loaded_arrays['a']
y_train = loaded_arrays['b']
X_val = loaded_arrays['c']
y_val = loaded_arrays['d']
X_test = loaded_arrays['e']
y_test = loaded_arrays['f']
y_test
loaded_arrays.close()

In [118]:
# ============================================================
# 0) Configuration
# ============================================================

#SEED = 42
#SEED = 31
SEED = 7

tf.keras.utils.set_random_seed(SEED)

# Optional:
# This requests deterministic TensorFlow operations where supported.
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass


OUTPUT_DIR = Path("cnn_21_classes_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

#choose the right input_shape value for the input data used

model_config = {
    "input_shape": (1000, 1),
    "blocks": 4,
    "base_filters": 16,
    "kernel_size": 20,
    "dilation_mode": "exp",
    "dropout": 0.0,
    "head_units": 32,
    "use_batchnorm": True,
    "pooling_head": "gap",
    "kernel_initializer": "he_normal",
    "learning_rate": 0.5e-4,
    "batch_size": 64,
    "max_epochs": 300,
    "early_stopping_patience": 10,
    "random_seed": SEED,
    "num_classes": 21
}


# Replace this with your own class weights.
# Set class_weight = None if you do not want class weighting.
class_weight = {0: 0.43199639477242,
 1: 2.1264418811002663,
 2: 19.01984126984127,
 3: 15.92358803986711,
 4: 7.058910162002945,
 5: 11.605326876513317,
 6: 1.1991493620215161,
 7: 3.741608118657299,
 8: 9.781632653061225,
 9: 8.05546218487395,
 10: 3.5848915482423336,
 11: 3.2450914014895056,
 12: 4.504699248120301,
 13: 2.1599819738621,
 14: 2.3774801587301586,
 15: 0.10019231573226306,
 16: 2.7833914053426247,
 17: 7.132440476190476,
 18: 13.694285714285714,
 19: 2.535978835978836,
 20: 0.2561594783817006}


In [120]:

# ============================================================
# 1) Prepare labels
# ============================================================

def prepare_multiclass_labels(y, num_classes=4):
    """
    Convert labels to a one-dimensional integer array.

    Expected class labels:
    0, 1, ..., num_classes - 1

    Supported input shapes:
    - (N,)
    - (N, 1)
    """
    y = np.asarray(y)

    if y.ndim == 2 and y.shape[1] == 1:
        y = y.ravel()

    elif y.ndim != 1:
        raise ValueError(
            "Multi-class labels must have shape (N,) or (N, 1). "
            f"Received shape: {y.shape}"
        )

    y = y.astype(np.int32)

    unique_labels = np.unique(y)
    expected_labels = np.arange(num_classes)

    if not np.all(np.isin(unique_labels, expected_labels)):
        raise ValueError(
            f"Labels must be integers between 0 and {num_classes - 1}. "
            f"Found labels: {unique_labels}"
        )

    return y


num_classes = model_config.get("num_classes", 21)

y_train_int = prepare_multiclass_labels(
    y_train,
    num_classes=num_classes
)

y_val_int = prepare_multiclass_labels(
    y_val,
    num_classes=num_classes
)

y_test_int = prepare_multiclass_labels(
    y_test,
    num_classes=num_classes
)

X_train_model = np.asarray(X_train, dtype=np.float32)
X_val_model = np.asarray(X_val, dtype=np.float32)
X_test_model = np.asarray(X_test, dtype=np.float32)


# Add a channel dimension if data have shape (N, 1000).
if X_train_model.ndim == 2:
    X_train_model = X_train_model[..., np.newaxis]

if X_val_model.ndim == 2:
    X_val_model = X_val_model[..., np.newaxis]

if X_test_model.ndim == 2:
    X_test_model = X_test_model[..., np.newaxis]


if X_train_model.shape[1:] != model_config["input_shape"]:
    raise ValueError(
        f"X_train has model input shape {X_train_model.shape[1:]}, "
        f"but expected {model_config['input_shape']}."
    )

if X_val_model.shape[1:] != model_config["input_shape"]:
    raise ValueError(
        f"X_val has model input shape {X_val_model.shape[1:]}, "
        f"but expected {model_config['input_shape']}."
    )


if X_test_model.shape[1:] != model_config["input_shape"]:
    raise ValueError(
        f"X_tst has model input shape {X_test_model.shape[1:]}, "
        f"but expected {model_config['input_shape']}."
    )


if len(X_train_model) != len(y_train_int):
    raise ValueError(
        "X_train and y_train contain different numbers of samples: "
        f"{len(X_train_model)} versus {len(y_train_int)}."
    )

if len(X_val_model) != len(y_val_int):
    raise ValueError(
        "X_val and y_val contain different numbers of samples: "
        f"{len(X_val_model)} versus {len(y_val_int)}."
    )

if len(X_test_model) != len(y_test_int):
    raise ValueError(
        "X_test and y_test contain different numbers of samples: "
        f"{len(X_test_model)} versus {len(y_test_int)}."
    )


In [122]:

# ============================================================
# 2) CNN model builder
# ============================================================

def build_binary_cnn(
    input_shape=(1000, 1),
    blocks=4,
    base_filters=32,
    kernel_size=20,
    dilation_mode="exp",
    dropout=0.0,
    head_units=32,
    use_batchnorm=True,
    pooling_head="gap",
    kernel_initializer="he_normal",
    num_classes=21
):
    inputs = tf.keras.Input(
        shape=input_shape,
        name="ecg_input"
    )

    x = inputs

    for block_index in range(blocks):

        # Filter schedule:
        # block 1: 32 filters
        # block 2: 64 filters
        # block 3: 128 filters
        # block 4: 128 filters
        filters = base_filters * (2 ** min(block_index, 2))

        if dilation_mode == "exp":
            dilation_rate = 2 ** block_index
        elif dilation_mode == "none":
            dilation_rate = 1
        else:
            raise ValueError(
                "dilation_mode must be either 'none' or 'exp'."
            )

        x = tf.keras.layers.Conv1D(
            filters=filters,
            kernel_size=kernel_size,
            dilation_rate=dilation_rate,
            padding="same",
            kernel_initializer=kernel_initializer,
            use_bias=not use_batchnorm,
            name=f"conv_block_{block_index + 1}"
        )(x)

        if use_batchnorm:
            x = tf.keras.layers.BatchNormalization(
                name=f"batchnorm_block_{block_index + 1}"
            )(x)

        x = tf.keras.layers.Activation(
            "relu",
            name=f"relu_block_{block_index + 1}"
        )(x)

        x = tf.keras.layers.MaxPooling1D(
            pool_size=2,
            name=f"maxpool_block_{block_index + 1}"
        )(x)

        if dropout > 0:
            x = tf.keras.layers.Dropout(
                rate=dropout,
                name=f"dropout_block_{block_index + 1}"
            )(x)

    if pooling_head == "gap":
        x = tf.keras.layers.GlobalAveragePooling1D(
            name="global_average_pooling"
        )(x)

    elif pooling_head == "flatten":
        x = tf.keras.layers.Flatten(
            name="flatten"
        )(x)

    else:
        raise ValueError(
            "pooling_head must be either 'gap' or 'flatten'."
        )

    if head_units > 0:
        x = tf.keras.layers.Dense(
            units=head_units,
            activation="relu",
            kernel_initializer=kernel_initializer,
            name="dense_head"
        )(x)

        if dropout > 0:
            x = tf.keras.layers.Dropout(
                rate=dropout,
                name="dense_head_dropout"
            )(x)

    outputs = tf.keras.layers.Dense(
        num_classes,
        activation="softmax",
        name="21_class_output"
    )(x)

    model = tf.keras.Model(
        inputs=inputs,
        outputs=outputs,
        name="21_class_ecg_cnn"
    )

    return model


model = build_binary_cnn(
    input_shape=model_config["input_shape"],
    blocks=model_config["blocks"],
    base_filters=model_config["base_filters"],
    kernel_size=model_config["kernel_size"],
    dilation_mode=model_config["dilation_mode"],
    dropout=model_config["dropout"],
    head_units=model_config["head_units"],
    use_batchnorm=model_config["use_batchnorm"],
    pooling_head=model_config["pooling_head"],
    kernel_initializer=model_config["kernel_initializer"]
)


In [124]:
# ============================================================
# 3) Compile model
# ============================================================

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=model_config["learning_rate"]
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        )
    ]
)


# ============================================================
# 4) Store architecture and model summary
# ============================================================

with open(
    OUTPUT_DIR / "model_configuration.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        model_config,
        file,
        indent=4,
        default=str
    )


with open(
    OUTPUT_DIR / "model_summary.txt",
    "w",
    encoding="utf-8"
) as file:
    model.summary(
        print_fn=lambda line: file.write(line + "\n")
    )


# Store the complete Keras model architecture as JSON.
with open(
    OUTPUT_DIR / "model_architecture.json",
    "w",
    encoding="utf-8"
) as file:
    file.write(model.to_json())



In [126]:
# ============================================================
# 5) Callbacks
# ============================================================

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=model_config["early_stopping_patience"],
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ModelCheckpoint(
        filepath=OUTPUT_DIR / "best_21_class_cnn.keras",
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.CSVLogger(
        filename=OUTPUT_DIR / "training_log.csv",
        separator=",",
        append=False
    ),

    tf.keras.callbacks.TerminateOnNaN()
]


In [128]:
# ============================================================
# 6) Train model
# ============================================================

history = model.fit(
    X_train_model,
    y_train_int,
    validation_data=(
        X_val_model,
        y_val_int
    ),
    epochs=model_config["max_epochs"],
    batch_size=model_config["batch_size"],
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1,
    shuffle=True
)


# ============================================================
# 7) Store training history
# ============================================================

history_df = pd.DataFrame(history.history)
history_df.index = np.arange(1, len(history_df) + 1)
history_df.index.name = "epoch"

history_df.to_csv(
    OUTPUT_DIR / "training_history.csv"
)


# ============================================================
# 8) Generate class predictions
# ============================================================

train_probabilities = model.predict(
    X_train_model,
    batch_size=model_config["batch_size"],
    verbose=0
)

val_probabilities = model.predict(
    X_val_model,
    batch_size=model_config["batch_size"],
    verbose=0
)

test_probabilities = model.predict(
    X_test_model,
    batch_size=model_config["batch_size"],
    verbose=0
)


train_predictions = np.argmax(
    train_probabilities,
    axis=1
).astype(np.int32)

val_predictions = np.argmax(
    val_probabilities,
    axis=1
).astype(np.int32)

test_predictions = np.argmax(
    test_probabilities,
    axis=1
).astype(np.int32)

# ============================================================
# 9) Calculate final metrics
# ============================================================

train_loss, train_keras_accuracy = model.evaluate(
    X_train_model,
    y_train_int,
    batch_size=model_config["batch_size"],
    verbose=0
)

val_loss, val_keras_accuracy = model.evaluate(
    X_val_model,
    y_val_int,
    batch_size=model_config["batch_size"],
    verbose=0
)

test_loss, test_keras_accuracy = model.evaluate(
    X_test_model,
    y_test_int,
    batch_size=model_config["batch_size"],
    verbose=0
)


train_accuracy = accuracy_score(
    y_train_int,
    train_predictions
)

val_accuracy = accuracy_score(
    y_val_int,
    val_predictions
)

test_accuracy = accuracy_score(
    y_test_int,
    test_predictions
)


train_f1 = f1_score(
    y_train_int,
    train_predictions,
    average="macro",
    zero_division=0
)

val_f1 = f1_score(
    y_val_int,
    val_predictions,
    average="macro",
    zero_division=0
)

test_f1 = f1_score(
    y_test_int,
    test_predictions,
    average="macro",
    zero_division=0
)


train_confusion_matrix = confusion_matrix(
    y_train_int,
    train_predictions,
    labels=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
)


val_confusion_matrix = confusion_matrix(
    y_val_int,
    val_predictions,
    labels=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
)

test_confusion_matrix = confusion_matrix(
    y_test_int,
    test_predictions,
    labels=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
)


# ============================================================
# 10) Store final metrics
# ============================================================

final_metrics = {
    "num_classes": model_config["num_classes"],
    "number_of_model_parameters": int(model.count_params()),

    "training": {
        "loss": float(train_loss),
        "accuracy": float(train_accuracy),
        "keras_accuracy": float(train_keras_accuracy),
        "macro_f1_score": float(train_f1)
    },

    "validation": {
        "loss": float(val_loss),
        "accuracy": float(val_accuracy),
        "keras_accuracy": float(val_keras_accuracy),
        "macro_f1_score": float(val_f1)
    },
    "test": {
        "loss": float(test_loss),
        "accuracy": float(test_accuracy),
        "keras_accuracy": float(test_keras_accuracy),
        "macro_f1_score": float(test_f1)
    }  
}


with open(
    OUTPUT_DIR / "final_metrics.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        final_metrics,
        file,
        indent=4
    )


# Also store the main metrics in table form.
metrics_df = pd.DataFrame(
    {
        "dataset": ["training", "validation", "test"],
        "loss": [train_loss, val_loss, test_loss],
        "accuracy": [train_accuracy, val_accuracy, test_accuracy],
        "macro_f1_score": [train_f1, val_f1, test_f1]
    }
)

metrics_df.to_csv(
    OUTPUT_DIR / "final_metrics.csv",
    index=False
)


# ============================================================
# 11) Store confusion matrices
# ============================================================

confusion_labels = [
    "predicted_class_0",
    "predicted_class_1",
    "predicted_class_2",
    "predicted_class_3",
    "predicted_class_4",
    "predicted_class_5",
    "predicted_class_6",
    "predicted_class_7",
    "predicted_class_8",
    "predicted_class_9",
    "predicted_class_10",
    "predicted_class_11",
    "predicted_class_12",
    "predicted_class_13",
    "predicted_class_14",
    "predicted_class_15",
    "predicted_class_16",
    "predicted_class_17",
    "predicted_class_18",
    "predicted_class_19",
    "predicted_class_20"
]

confusion_index = [
    "predicted_class_0",
    "predicted_class_1",
    "predicted_class_2",
    "predicted_class_3",
    "predicted_class_4",
    "predicted_class_5",
    "predicted_class_6",
    "predicted_class_7",
    "predicted_class_8",
    "predicted_class_9",
    "predicted_class_10",
    "predicted_class_11",
    "predicted_class_12",
    "predicted_class_13",
    "predicted_class_14",
    "predicted_class_15",
    "predicted_class_16",
    "predicted_class_17",
    "predicted_class_18",
    "predicted_class_19",
    "predicted_class_20"
]


train_cm_df = pd.DataFrame(
    train_confusion_matrix,
    index=confusion_index,
    columns=confusion_labels
)

val_cm_df = pd.DataFrame(
    val_confusion_matrix,
    index=confusion_index,
    columns=confusion_labels
)

test_cm_df = pd.DataFrame(
    test_confusion_matrix,
    index=confusion_index,
    columns=confusion_labels
)


train_cm_df.to_csv(
    OUTPUT_DIR / "training_confusion_matrix.csv"
)

val_cm_df.to_csv(
    OUTPUT_DIR / "validation_confusion_matrix.csv"
)

test_cm_df.to_csv(
    OUTPUT_DIR / "test_confusion_matrix.csv"
)
# ============================================================
# 12) Store detailed classification reports
# ============================================================

train_report = classification_report(
    y_train_int,
    train_predictions,
    labels=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20],
    target_names=[
        "class_0",
        "class_1",
        "class_2",
        "class_3",
        "class_4",
        "class_5",
        "class_6",
        "class_7",
        "class_8",
        "class_9",
        "class_10",
        "class_11",
        "class_12",
        "class_13",
        "class_14",
        "class_15",
        "class_16",
        "class_17",
        "class_18",
        "class_19",
        "class_20"
    ],
    output_dict=True,
    zero_division=0
)

val_report = classification_report(
    y_val_int,
    val_predictions,
    labels=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20],
    target_names=[
        "class_0",
        "class_1",
        "class_2",
        "class_3",
        "class_4",
        "class_5",
        "class_6",
        "class_7",
        "class_8",
        "class_9",
        "class_10",
        "class_11",
        "class_12",
        "class_13",
        "class_14",
        "class_15",
        "class_16",
        "class_17",
        "class_18",
        "class_19",
        "class_20"
    ],
    output_dict=True,
    zero_division=0
)

test_report = classification_report(
    y_test_int,
    test_predictions,
    labels=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20],
    target_names=[
        "class_0",
        "class_1",
        "class_2",
        "class_3",
        "class_4",
        "class_5",
        "class_6",
        "class_7",
        "class_8",
        "class_9",
        "class_10",
        "class_11",
        "class_12",
        "class_13",
        "class_14",
        "class_15",
        "class_16",
        "class_17",
        "class_18",
        "class_19",
        "class_20"
    ],
    output_dict=True,
    zero_division=0
)

train_report_df = pd.DataFrame(train_report).transpose()
val_report_df = pd.DataFrame(val_report).transpose()
test_report_df = pd.DataFrame(test_report).transpose()


train_report_df.to_csv(
    OUTPUT_DIR / "training_classification_report.csv"
)

val_report_df.to_csv(
    OUTPUT_DIR / "validation_classification_report.csv"
)

test_report_df.to_csv(
    OUTPUT_DIR / "test_classification_report.csv"
)


# ============================================================
# 13) Store predictions and probabilities
# ============================================================

train_prediction_data = {
    "true_label": y_train_int,
    "predicted_label": train_predictions
}

val_prediction_data = {
    "true_label": y_val_int,
    "predicted_label": val_predictions
}

test_prediction_data = {
    "true_label": y_test_int,
    "predicted_label": test_predictions
}


for class_index in range(model_config["num_classes"]):
    train_prediction_data[
        f"probability_class_{class_index}"
    ] = train_probabilities[:, class_index]

    val_prediction_data[
        f"probability_class_{class_index}"
    ] = val_probabilities[:, class_index]

    test_prediction_data[
        f"probability_class_{class_index}"
    ] = test_probabilities[:, class_index]

train_prediction_df = pd.DataFrame(train_prediction_data)
val_prediction_df = pd.DataFrame(val_prediction_data)
test_prediction_df = pd.DataFrame(test_prediction_data)

train_prediction_df.to_csv(
    OUTPUT_DIR / "training_predictions.csv",
    index=False
)

val_prediction_df.to_csv(
    OUTPUT_DIR / "validation_predictions.csv",
    index=False
)

test_prediction_df.to_csv(
    OUTPUT_DIR / "test_predictions.csv",
    index=False
)
# ============================================================
# 14) Save final model
# ============================================================

model.save(
    OUTPUT_DIR / "final_21_class_cnn.keras"
)


# ============================================================
# 15) Print summary
# ============================================================

print("\nTraining completed.")
print(f"Results directory: {OUTPUT_DIR.resolve()}")
print(f"Number of parameters: {model.count_params():,}")

print("\nTraining metrics:")
print(f"Loss:     {train_loss:.4f}")
print(f"Accuracy: {train_accuracy:.4f}")
print(f"F1-score: {train_f1:.4f}")
print("Confusion matrix:")
print(train_confusion_matrix)

print("\nTraining classification report:")
print(train_report_df)

print("\nValidation metrics:")
print(f"Loss:     {val_loss:.4f}")
print(f"Accuracy: {val_accuracy:.4f}")
print(f"F1-score: {val_f1:.4f}")
print("Confusion matrix:")
print(val_confusion_matrix)

print("\nValidation classification report:")
print(val_report_df)

print("\nTest metrics:")
print(f"Loss:     {test_loss:.4f}")
print(f"Accuracy: {test_accuracy:.4f}")
print(f"F1-score: {test_f1:.4f}")
print("Confusion matrix:")
print(test_confusion_matrix)

print("\nTest classification report:")
print(test_report_df)


Epoch 1/300
225/225 ━━━━━━━━━━━━━━━━━━━━ 0s 138ms/step - accuracy: 0.0197 - loss: 3.2656
Epoch 1: val_loss improved from None to 3.25377, saving model to cnn_21_classes_results\best_21_class_cnn.keras
225/225 ━━━━━━━━━━━━━━━━━━━━ 41s 153ms/step - accuracy: 0.0225 - loss: 3.1446 - val_accuracy: 0.0200 - val_loss: 3.2538
Epoch 2/300
225/225 ━━━━━━━━━━━━━━━━━━━━ 0s 157ms/step - accuracy: 0.0376 - loss: 2.9765
Epoch 2: val_loss improved from 3.25377 to 3.16822, saving model to cnn_21_classes_results\best_21_class_cnn.keras
225/225 ━━━━━━━━━━━━━━━━━━━━ 38s 170ms/step - accuracy: 0.0370 - loss: 2.9422 - val_accuracy: 0.0390 - val_loss: 3.1682
Epoch 3/300
225/225 ━━━━━━━━━━━━━━━━━━━━ 0s 148ms/step - accuracy: 0.0441 - loss: 2.8816
Epoch 3: val_loss improved from 3.16822 to 3.05961, saving model to cnn_21_classes_results\best_21_class_cnn.keras
225/225 ━━━━━━━━━━━━━━━━━━━━ 39s 161ms/step - accuracy: 0.0440 - loss: 2.8556 - val_accuracy: 0.0472 - val_loss: 3.0596
Epoch 4/300
225/225 ━━━━━━━━━━━